## 1. 라이브러리 및 환경 설정


In [1]:
# 1-1. 필요한 패키지 설치

!pip install numpy
!pip install pandas
!pip install lightgbm
!pip install scikit-learn
!pip install pyarrow
!pip install fastparquet
!pip install xgboost
!pip install catboost
!pip install optuna


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# 1-2. 라이브러리 임포트 및 경고 무시 설정

import warnings
from pathlib import Path
import os

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor, callback # <<< [수정] XGBoost EarlyStopping 콜백 임포트
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import pyarrow
import fastparquet
import optuna


warnings.filterwarnings("ignore")

/Users/leetae04kr/develope/ETC_DACON/스마트창고출고지연예측/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 데이터 로드


In [3]:
# 2-1. 데이터 경로 설정 및 CSV 파일 로드

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists() and (ROOT.parent / "data" / "train.csv").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
print(f"데이터 경로: {DATA_DIR}")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
layout = pd.read_csv(DATA_DIR / "layout_info.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")
print(f"레이아웃 데이터 크기: {layout.shape}")

데이터 경로: /Users/leetae04kr/develope/ETC_DACON/스마트창고출고지연예측/data
학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)
레이아웃 데이터 크기: (300, 15)


## 3. 피처 및 타겟 설정


In [4]:
# 3-1. 타겟 변수 및 식별자(ID) 변수 설정

TARGET = "avg_delay_minutes_next_30m"
ID_COL = "ID"



In [5]:
# 3-2. 피처 엔지니어링 함수 정의 (build_features)

def build_features(df: pd.DataFrame, layout_df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    out = df.copy()

    # layout_id 기준으로 보조 테이블 결합
    out = out.merge(layout_df, on="layout_id", how="left")

    # 시나리오 단위 시계열 생성 전 정렬
    out = out.sort_values(["scenario_id", ID_COL]).copy()
    grp = out.groupby("scenario_id", sort=False)

    # 시나리오 진행 구간 파생 변수
    out["scenario_step_idx"] = grp.cumcount()
    out["scenario_step_cnt"] = grp["scenario_id"].transform("size")
    out["scenario_progress"] = out["scenario_step_idx"] / np.maximum(out["scenario_step_cnt"] - 1, 1)

    eps = 1e-6

    # 1) 시계열(Temporal) 피처: lag / diff / rolling
    lag_targets = [
        "congestion_score", "order_inflow_15m", "robot_utilization", "battery_mean",
        "charge_queue_length", "fault_count_15m", "wms_response_time_ms", "pack_utilization",
        "loading_dock_util", "staff_on_floor", "worker_avg_tenure_months"
    ]
    lag_periods = [1, 2, 3]
    diff_periods = [1, 2]
    
    for col in lag_targets:
        if col in out.columns:
            for lag in lag_periods:
                out[f"{col}_lag{lag}"] = grp[col].shift(lag)
            for diff_p in diff_periods:
                out[f"{col}_diff{diff_p}"] = grp[col].diff(diff_p)

    rolling_targets = ["order_inflow_15m", "congestion_score", "robot_utilization"]
    rolling_windows = [3, 5]
    for col in rolling_targets:
        if col in out.columns:
            for window in rolling_windows:
                out[f"{col}_roll{window}_mean"] = grp[col].transform(lambda s: s.rolling(window=window, min_periods=1).mean())
                out[f"{col}_roll{window}_std"] = grp[col].transform(lambda s: s.rolling(window=window, min_periods=1).std())

    # 2) 로봇 운영 효율(Efficiency) 지표
    if {"robot_active", "robot_idle"}.issubset(out.columns):
        out["effective_robot_utilization"] = out["robot_active"] / (out["robot_active"] + out["robot_idle"] + eps)

    if {"low_battery_ratio", "avg_charge_wait"}.issubset(out.columns):
        out["charging_pressure"] = out["low_battery_ratio"] * out["avg_charge_wait"]

    if {"order_inflow_15m", "robot_active"}.issubset(out.columns):
        out["task_density_per_robot"] = out["order_inflow_15m"] / (out["robot_active"] + eps)

    # 3) 공간 및 인프라(Spatial & Infra)
    area_candidates = ["warehouse_area", "warehouse_area_m2", "layout_area", "area_m2", "total_area"]
    area_col = next((c for c in area_candidates if c in out.columns), None)
    if area_col is not None and "congestion_score" in out.columns:
        out["congestion_per_area"] = out["congestion_score"] / (out[area_col] + eps)

    if {"max_zone_density", "aisle_traffic_score"}.issubset(out.columns):
        out["bottleneck_occupancy"] = out["max_zone_density"] * out["aisle_traffic_score"]

    if {"wifi_signal_db", "network_latency_ms"}.issubset(out.columns):
        out["it_reliability_raw"] = out["wifi_signal_db"] * out["network_latency_ms"]

    # 4) 환경/작업자(Soft Factors)
    if {"pack_utilization", "staff_on_floor"}.issubset(out.columns):
        out["pack_bottleneck_per_staff"] = out["pack_utilization"] / (out["staff_on_floor"] + eps)

    if {"pack_utilization", "staff_on_floor", "worker_avg_tenure_months"}.issubset(out.columns):
        out["pack_bottleneck_tenure_adjusted"] = out["pack_utilization"] / (
            (out["staff_on_floor"] + eps) * (1.0 + out["worker_avg_tenure_months"] / 12.0)
        )

    if {"fleet_age_months_avg", "fault_count_15m"}.issubset(out.columns):
        out["device_fatigue_index"] = out["fleet_age_months_avg"] * out["fault_count_15m"]

    # 기존 상호작용 유지
    if {"robot_active", "robot_idle", "robot_charging"}.issubset(out.columns):
        total_robot = out["robot_active"] + out["robot_idle"] + out["robot_charging"] + eps
        out["robot_active_ratio"] = out["robot_active"] / total_robot
        out["robot_charging_ratio"] = out["robot_charging"] / total_robot

    if {"charge_queue_length", "robot_charging"}.issubset(out.columns):
        out["charge_queue_per_charging_robot"] = out["charge_queue_length"] / (out["robot_charging"] + eps)

    if {"congestion_score", "aisle_traffic_score"}.issubset(out.columns):
        out["traffic_congestion_interaction"] = out["congestion_score"] * out["aisle_traffic_score"]

    if {"pack_utilization", "loading_dock_util"}.issubset(out.columns):
        out["packing_loading_gap"] = out["pack_utilization"] - out["loading_dock_util"]

    if {"network_latency_ms", "wms_response_time_ms"}.issubset(out.columns):
        out["it_latency_total"] = out["network_latency_ms"] + out["wms_response_time_ms"]

    # 시계열 파생의 결측은 시나리오 내부로 보정
    new_cols = [c for c in out.columns if c not in df.columns and c not in layout_df.columns]
    for col in new_cols:
        if out[col].isna().any():
            out[col] = grp[col].transform(lambda s: s.ffill().bfill())
            out[col] = out[col].fillna(out[col].median()) # 중앙값으로 채우기

    # 메모리 최적화
    float_cols = out.select_dtypes(include=['float64']).columns
    out[float_cols] = out[float_cols].astype('float32')

    # 예측 제출 정합성을 위해 원래 ID 순서 복원
    out = out.sort_values(ID_COL).reset_index(drop=True)
    return out


In [6]:
# 3-3. 피처 저장 폴더 및 강제 업데이트 옵션 설정


# --- 설정 ---
FEATURE_DIR = ROOT / "features" # 피처를 저장할 폴더
FORCE_UPDATE = False     # True로 바꾸면 기존 파일을 무시하고 새로 피처를 만듭니다.

# 폴더가 없으면 생성
FEATURE_DIR.mkdir(exist_ok=True)


In [7]:
# 3-4. 피처 생성 및 불러오기 함수 정의 (get_features)


def get_features(train, test, layout, force_update=False):
    train_path = FEATURE_DIR / "train_feat.parquet"
    test_path = FEATURE_DIR / "test_feat.parquet"
    
    cat_cols = ["layout_id", "scenario_id"]

    # 기존에 저장된 파일이 있고, 강제 업데이트 모드가 아니면 로드
    if not force_update and train_path.exists() and test_path.exists():
        print("💾 저장된 피처 파일을 불러옵니다...")
        train_feat = pd.read_parquet(train_path)
        test_feat = pd.read_parquet(test_path)
        
        # 카테고리 타입 복원
        for c in cat_cols:
            if c in train_feat.columns:
                train_feat[c] = train_feat[c].astype("category")
            if c in test_feat.columns:
                test_feat[c] = test_feat[c].astype("category")

    else:
        print("🛠️ 피처 생성을 시작합니다 (이 작업은 시간이 소요될 수 있습니다)...")
        train_feat = build_features(train, layout, is_train=True)
        test_feat = build_features(test, layout, is_train=False)

        # to_parquet 저장 전 카테고리 타입 변경
        for c in cat_cols:
            if c in train_feat.columns:
                train_feat[c] = train_feat[c].astype(object)
            if c in test_feat.columns:
                test_feat[c] = test_feat[c].astype(object)

        # 결과 저장
        train_feat.to_parquet(train_path, index=False, engine='pyarrow')
        test_feat.to_parquet(test_path, index=False, engine='pyarrow')
        print(f"✅ 피처 생성 및 저장 완료: {FEATURE_DIR}")

        # 카테고리 타입 복원
        for c in cat_cols:
            if c in train_feat.columns:
                train_feat[c] = train_feat[c].astype("category")
            if c in test_feat.columns:
                test_feat[c] = test_feat[c].astype("category")

    return train_feat, test_feat


In [8]:
# 3-5. 피처 데이터셋 준비 및 사용할 피처 리스트 정리


# --- 실행 ---
train_feat, test_feat = get_features(train, test, layout, force_update=FORCE_UPDATE)

ID_COLS = [ID_COL]
DROP_COLS = ID_COLS + ([TARGET] if TARGET in train_feat.columns else [])
feature_cols = [c for c in train_feat.columns if c not in DROP_COLS]

print(f"최종 피처 수: {len(feature_cols)}")
print("레이아웃 결합 후 train shape:", train_feat.shape)
print("레이아웃 결합 후 test shape:", test_feat.shape)

💾 저장된 피처 파일을 불러옵니다...
최종 피처 수: 132
레이아웃 결합 후 train shape: (250000, 134)
레이아웃 결합 후 test shape: (50000, 133)


## 4. 모델 학습 (5-Fold CV)


In [9]:
# 4-1. 범주형 컬럼 정의 및 카테고리 레벨 통일

# 1. 범주형 컬럼 정의
cat_cols = [c for c in ["layout_id", "scenario_id", "layout_type"] if c in feature_cols]

# 2. 학습/테스트 데이터의 카테고리 레벨을 하나로 통일합니다.
for col in cat_cols:
    # 전체 데이터(학습+테스트)에 존재하는 모든 고유값을 찾습니다.
    combined_categories = pd.concat([train_feat[col], test_feat[col]]).astype(str).unique()
    
    # 두 데이터셋 모두 동일한 카테고리 레벨을 갖도록 설정합니다.
    train_feat[col] = pd.Categorical(train_feat[col].astype(str), categories=combined_categories)
    test_feat[col] = pd.Categorical(test_feat[col].astype(str), categories=combined_categories)

print(f"✅ 범주형 변수({cat_cols})의 레벨 통일 완료")

# --- Optuna를 사용한 하이퍼파라미터 최적화 ---


✅ 범주형 변수(['layout_id', 'scenario_id', 'layout_type'])의 레벨 통일 완료


In [10]:
# 4-2. Optuna 하이퍼파라미터 최적화 목적 함수 (objective)

def objective(trial, data, target_col, feature_cols, cat_cols):
    # 최적화할 하이퍼파라미터 범위 정의
    params = {
        'objective': 'mae',
        'metric': 'mae',
        'n_estimators': trial.suggest_int('n_estimators', 1000, 5000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }

    # 최적화 시간을 줄이기 위해 데이터의 20%만 샘플링하여 사용
    sampled_data = data.sample(frac=0.2, random_state=42)
    groups = sampled_data["scenario_id"]
    gkf = GroupKFold(n_splits=3) # CV Fold 수도 줄여서 속도 확보
    
    oof_preds = np.zeros(len(sampled_data))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(sampled_data, sampled_data[target_col], groups=groups)):
        X_tr = sampled_data.iloc[tr_idx][feature_cols]
        y_tr = sampled_data.iloc[tr_idx][target_col]
        X_val = sampled_data.iloc[val_idx][feature_cols]
        y_val = sampled_data.iloc[val_idx][target_col]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)],
            eval_metric='l1',
            categorical_feature=cat_cols if len(cat_cols) > 0 else "auto"
        )
        oof_preds[val_idx] = model.predict(X_val)

    mae = mean_absolute_error(sampled_data[target_col], oof_preds)
    return mae


In [11]:
# 4-3. Optuna 최적화 실행 및 최종 파라미터 결정

# --- 최적화 실행 ---
# 최적화를 수행할지 여부 결정 (시간이 오래 걸릴 수 있으므로 필요할 때만 True로 변경)
RUN_OPTUNA = True 

if RUN_OPTUNA:
    print("🤖 Optuna를 사용한 하이퍼파라미터 최적화를 시작합니다...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, train_feat, TARGET, feature_cols, cat_cols), n_trials=50) # n_trials로 시도 횟수 조절

    print("✅ 최적화 완료!")
    print(f"최적 MAE: {study.best_value:.5f}")
    print("최적 하이퍼파라미터:")
    print(study.best_params)
    
    # 찾은 최적 파라미터를 기본 모델 파라미터에 업데이트
    best_params = study.best_params
else:
    print("⏩ Optuna 최적화를 건너뛰고 기존 파라미터로 학습합니다.")
    best_params = {
        "objective": "mae",
        "n_estimators": 4000,
        "learning_rate": 0.02,
        "num_leaves": 127,
        "max_depth": -1,
        "min_child_samples": 60,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.3,
        "reg_lambda": 1.0,
    }

# 기본 파라미터 설정
base_lgbm_params = {
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}
# 최적화된 파라미터와 기본 파라미터를 결합
final_lgbm_params = {**base_lgbm_params, **best_params}


[I 2026-05-02 00:50:37,669] A new study created in memory with name: no-name-b196c854-4a56-42a3-91c2-773ab886db7c


🤖 Optuna를 사용한 하이퍼파라미터 최적화를 시작합니다...


[I 2026-05-02 00:50:40,011] Trial 0 finished with value: 9.473760277804825 and parameters: {'n_estimators': 2000, 'learning_rate': 0.0826781994942673, 'num_leaves': 108, 'max_depth': 3, 'min_child_samples': 66, 'subsample': 0.8106990867090971, 'colsample_bytree': 0.9132563065686108, 'reg_alpha': 0.545785566620354, 'reg_lambda': 0.5754718921208701}. Best is trial 0 with value: 9.473760277804825.
[I 2026-05-02 00:51:05,077] Trial 1 finished with value: 9.350074480487425 and parameters: {'n_estimators': 1000, 'learning_rate': 0.012467091155169982, 'num_leaves': 51, 'max_depth': 11, 'min_child_samples': 79, 'subsample': 0.8268022781097494, 'colsample_bytree': 0.7926828381245887, 'reg_alpha': 0.06460353186390366, 'reg_lambda': 0.627213646792309}. Best is trial 1 with value: 9.350074480487425.
[I 2026-05-02 00:51:08,586] Trial 2 finished with value: 9.42623647123617 and parameters: {'n_estimators': 5000, 'learning_rate': 0.05983084345946194, 'num_leaves': 203, 'max_depth': 5, 'min_child_samp

✅ 최적화 완료!
최적 MAE: 9.27720
최적 하이퍼파라미터:
{'n_estimators': 4000, 'learning_rate': 0.01371093022658739, 'num_leaves': 226, 'max_depth': 0, 'min_child_samples': 26, 'subsample': 0.9602028685016755, 'colsample_bytree': 0.7151875572000421, 'reg_alpha': 0.08293615493380874, 'reg_lambda': 0.9910995121319043}


In [16]:
# 4-4. 앙상블에 사용할 개별 모델 정의 (LGBM, XGB, CatBoost)

# --- 모델 설정 ---
models = {
    "lgbm": LGBMRegressor(**final_lgbm_params),
    "xgb": XGBRegressor(
        objective='reg:squarederror',
        eval_metric='mae',
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),
    "cat": CatBoostRegressor(
        loss_function='MAE',
        iterations=3000,
        learning_rate=0.04,
        depth=8,
        l2_leaf_reg=3,
        random_seed=42,
        verbose=0,
        thread_count=-1
    )
}


In [17]:
# 4-5. K-Fold 교차 검증 기반 모델 학습 및 예측 실행

# --- 학습 설정 ---
groups = train_feat["scenario_id"]
gkf = GroupKFold(n_splits=5)

oof_preds = {}
test_preds = {}
model_maes = {}

# --- 개별 모델 학습 루프 ---
for model_name, model in models.items():
    print(f"\n{'='*10} Training {model_name.upper()} {'='*10}")
    
    model_oof = np.zeros(len(train_feat))
    model_test = np.zeros(len(test_feat))
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_feat, train_feat[TARGET], groups=groups), start=1):
        print(f"\n── Fold {fold} ──")

        X_tr = train_feat.iloc[tr_idx][feature_cols]
        y_tr = train_feat.iloc[tr_idx][TARGET]
        X_val = train_feat.iloc[val_idx][feature_cols]
        y_val = train_feat.iloc[val_idx][TARGET]

        # 모델별 학습 파라미터 설정
        if model_name == 'lgbm':
            fit_params = {
                "eval_set": [(X_val, y_val)],
                "callbacks": [lgb.early_stopping(200, verbose=False)],
                "eval_metric": "l1",
                "categorical_feature": cat_cols if len(cat_cols) > 0 else "auto"
            }
        elif model_name == 'xgb':
            # ==================================================================================
            # [수정] XGBoost EarlyStopping 방식 변경
            # ----------------------------------------------------------------------------------
            # 'early_stopping_rounds' 파라미터 대신 'callbacks'를 사용하여 조기 종료를 설정합니다.
            # ==================================================================================
            fit_params = {
                "eval_set": [(X_val, y_val)],
                # "callbacks": removed here, set via set_params instead
                "verbose": False
            }
        elif model_name == 'cat':
            fit_params = {
                "eval_set": [(X_val, y_val)],
                "early_stopping_rounds": 100,
                "verbose": 0,
                "cat_features": cat_cols
            }

        # 모델 복제 및 random_state 설정
        current_model = model.copy() if hasattr(model, 'copy') else model
        if hasattr(current_model, 'random_state'):
            current_model.random_state = 42 + fold
        elif hasattr(current_model, 'random_seed'):
             current_model.random_seed = 42 + fold
        
        if model_name == 'xgb':
            current_model.set_params(early_stopping_rounds=100)
        current_model.fit(X_tr, y_tr, **fit_params)

        # 예측
        val_pred = current_model.predict(X_val)
        tst_pred = current_model.predict(test_feat[feature_cols])

        model_oof[val_idx] = np.clip(val_pred, 0, None)
        model_test += np.clip(tst_pred, 0, None) / gkf.get_n_splits()

        fold_mae = mean_absolute_error(y_val, model_oof[val_idx])
        print(f"Fold {fold} MAE: {fold_mae:.5f}")

    # 모델별 결과 저장
    oof_preds[model_name] = model_oof
    test_preds[model_name] = model_test
    model_maes[model_name] = mean_absolute_error(train_feat[TARGET], model_oof)
    print(f"\n{model_name.upper()} OOF MAE: {model_maes[model_name]:.5f}")



========== Training LGBM ==========

── Fold 1 ──
Fold 1 MAE: 9.24711

── Fold 2 ──
Fold 2 MAE: 9.37318

── Fold 3 ──
Fold 3 MAE: 8.87260

── Fold 4 ──
Fold 4 MAE: 9.68589

── Fold 5 ──
Fold 5 MAE: 9.19371

LGBM OOF MAE: 9.27450

========== Training XGB ==========

── Fold 1 ──


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:layout_id: category, scenario_id: category, layout_type: category

## 5. 결과 확인


In [ ]:
# 5-1. 모델별 가중치 산출 및 최종 앙상블 결과(OOF) 확인

# 모델별 가중치 설정 (OOF MAE를 기반으로 역수에 비례하도록)
total_inverse_mae = sum(1 / mae for mae in model_maes.values())
weights = {name: (1 / mae) / total_inverse_mae for name, mae in model_maes.items()}

print("\n--- 모델별 가중치 ---")
for name, w in weights.items():
    print(f"{name.upper()}: {w:.4f}")

# 가중 평균 앙상블
final_test_preds = np.zeros(len(test_feat))
final_oof_preds = np.zeros(len(train_feat))

for name, w in weights.items():
    final_test_preds += test_preds[name] * w
    final_oof_preds += oof_preds[name] * w

ensemble_oof_mae = mean_absolute_error(train_feat[TARGET], final_oof_preds)
print(f"\n가중 평균 앙상블 OOF MAE: {ensemble_oof_mae:.5f}")

## 6. 제출 파일 생성


In [ ]:
# 6-1. 최종 예측 결과 저장 및 제출(submission) 파일 생성

pred_df = pd.DataFrame({ID_COL: test_feat[ID_COL].values, TARGET: final_test_preds})
submission = sample_sub[[ID_COL]].merge(pred_df, on=ID_COL, how="left")

save_path = ROOT / "submission_lgbm_only.csv"
submission.to_csv(save_path, index=False)
print(f"제출 파일 저장 완료: {save_path}")
submission.head()